
## Folder 06 — File 04: Referential Integrity Dashboard:



It will provide:

- Overall referential integrity KPIs.
- Relationship-level health monitoring.
- Orphan-record analysis.
- Null foreign-key analysis.
- Parent records without child records.
- SQL queries for Databricks dashboards.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
import uuid

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

RELATIONSHIP_REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.foreign_key_relationship_report"
)

CARDINALITY_REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.foreign_key_cardinality_report"
)

ORPHAN_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_orphans"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SCHEMA}.orphan_record_quarantine"
)

DASHBOARD_DETAIL_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_dashboard_detail"
)

DASHBOARD_SUMMARY_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_dashboard_summary"
)

RUN_ID = str(uuid.uuid4())

DASHBOARD_TIMESTAMP = datetime.now().isoformat(timespec="seconds")

print("Project configuration loaded.")
print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Run ID:", RUN_ID)
print("Generated at:", DASHBOARD_TIMESTAMP)


## 3. Validate input tables

REQUIRED_INPUT_TABLES = [
    RELATIONSHIP_REPORT_TABLE,
    CARDINALITY_REPORT_TABLE,
    ORPHAN_TABLE,
    QUARANTINE_TABLE
]

existing_tables = set(
    row.tableName
    for row in spark.sql(
        f"SHOW TABLES IN {CATALOG}.{SCHEMA}"
    ).collect()
)

missing_tables = []

for table_name in REQUIRED_INPUT_TABLES:

    short_name = table_name.split(".")[-1]

    if short_name in existing_tables:
        print(f"{table_name} -> EXISTS")
    else:
        print(f"{table_name} -> MISSING")
        missing_tables.append(table_name)

if missing_tables:
    raise Exception(
        f"Required input tables are missing: {missing_tables}"
    )

print("All input tables are available.")


## 4. Load source reports

relationship_df = spark.table(
    RELATIONSHIP_REPORT_TABLE
)

cardinality_df = spark.table(
    CARDINALITY_REPORT_TABLE
)

orphan_df = spark.table(
    ORPHAN_TABLE
)

quarantine_df = spark.table(
    QUARANTINE_TABLE
)

relationship_count = relationship_df.count()
cardinality_count = cardinality_df.count()
orphan_count = orphan_df.count()
quarantine_count = quarantine_df.count()

print("=" * 65)
print("SOURCE REPORT COUNTS")
print("=" * 65)

print("Relationship records:", relationship_count)
print("Cardinality records:", cardinality_count)
print("Orphan records:", orphan_count)
print("Quarantine records:", quarantine_count)

if relationship_count == 0:
    raise Exception("Relationship report contains no records.")

if cardinality_count == 0:
    raise Exception("Cardinality report contains no records.")

print("Source reports loaded successfully.")


## 5. Validate source schemas

REQUIRED_RELATIONSHIP_COLUMNS = [
    "relationship",
    "parent_dataset",
    "parent_key",
    "child_dataset",
    "child_key",
    "parent_records",
    "unique_parent_keys",
    "child_records",
    "null_child_foreign_keys",
    "orphan_child_records",
    "referenced_parent_records",
    "parents_without_children",
    "reference_match_rate",
    "parent_usage_rate"
]

REQUIRED_CARDINALITY_COLUMNS = [
    "relationship",
    "child_key_value",
    "child_record_count"
]

relationship_columns = relationship_df.columns
cardinality_columns = cardinality_df.columns

missing_relationship_columns = [
    column
    for column in REQUIRED_RELATIONSHIP_COLUMNS
    if column not in relationship_columns
]

missing_cardinality_columns = [
    column
    for column in REQUIRED_CARDINALITY_COLUMNS
    if column not in cardinality_columns
]

if missing_relationship_columns:
    raise Exception(
        "Missing relationship report columns: "
        + str(missing_relationship_columns)
    )

if missing_cardinality_columns:
    raise Exception(
        "Missing cardinality report columns: "
        + str(missing_cardinality_columns)
    )

print("Relationship report schema validated.")
print("Cardinality report schema validated.")

print("\nRelationship report schema:")
relationship_df.printSchema()

print("\nCardinality report schema:")
cardinality_df.printSchema()

print("Source schemas validated successfully.")


## 6. Prepare cardinality statistics

cardinality_summary_df = (
    cardinality_df
    .groupBy("relationship")
    .agg(

        F.count("*").alias(
            "cardinality_key_count"
        ),

        F.countDistinct(
            "child_key_value"
        ).alias(
            "cardinality_distinct_child_key_count"
        ),

        F.sum(
            "child_record_count"
        ).alias(
            "cardinality_child_record_count"
        ),

        F.max(
            "child_record_count"
        ).alias(
            "cardinality_maximum_children_per_key"
        ),

        F.round(
            F.avg("child_record_count"),
            2
        ).alias(
            "cardinality_average_children_per_key"
        ),

        F.min(
            "child_record_count"
        ).alias(
            "cardinality_minimum_children_per_key"
        )
    )
)

print("Cardinality statistics generated.")

display(
    cardinality_summary_df.orderBy("relationship")
)


## 7. Prepare relationship dashboard details

columns_to_remove = [
    "cardinality_key_count",
    "distinct_child_key_count",
    "cardinality_distinct_child_key_count",
    "cardinality_child_record_count",
    "cardinality_maximum_children_per_key",
    "cardinality_average_children_per_key",
    "cardinality_minimum_children_per_key",
    "average_children_per_key",
    "maximum_children_per_key",
    "minimum_children_per_key"
]

existing_columns = relationship_df.columns

relationship_base_df = relationship_df.drop(
    *[
        column
        for column in columns_to_remove
        if column in existing_columns
    ]
)

relationship_detail_df = (
    relationship_base_df

    .withColumn(
        "orphan_percentage",
        F.when(
            F.col("child_records") > 0,
            F.round(
                F.col("orphan_child_records")
                / F.col("child_records") * 100,
                2
            )
        ).otherwise(0.0)
    )

    .withColumn(
        "null_fk_percentage",
        F.when(
            F.col("child_records") > 0,
            F.round(
                F.col("null_child_foreign_keys")
                / F.col("child_records") * 100,
                2
            )
        ).otherwise(0.0)
    )

    .withColumn(
        "dashboard_status",
        F.when(
            F.col("orphan_child_records") > 0,
            "ORPHANS_FOUND"
        )
        .when(
            F.col("null_child_foreign_keys") > 0,
            "NULL_FOREIGN_KEYS_FOUND"
        )
        .otherwise("HEALTHY")
    )

    .withColumn(
        "dashboard_run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "dashboard_generated_at",
        F.lit(DASHBOARD_TIMESTAMP)
    )
)

# Join cardinality statistics using uniquely named columns.

relationship_detail_df = (
    relationship_detail_df
    .join(
        cardinality_summary_df,
        on="relationship",
        how="left"
    )
    .fillna({
        "cardinality_key_count": 0,
        "cardinality_distinct_child_key_count": 0,
        "cardinality_child_record_count": 0,
        "cardinality_maximum_children_per_key": 0,
        "cardinality_average_children_per_key": 0.0,
        "cardinality_minimum_children_per_key": 0
    })
)

print("Relationship dashboard details generated.")

print("Dashboard detail schema:")

relationship_detail_df.printSchema()

display(
    relationship_detail_df.orderBy("relationship")
)


## 8. Analyze orphan records

def find_column_case_insensitive(dataframe, candidates):

    column_lookup = {
        column.lower(): column
        for column in dataframe.columns
    }

    for candidate in candidates:

        if candidate.lower() in column_lookup:
            return column_lookup[candidate.lower()]

    return None


orphan_relationship_column = find_column_case_insensitive(
    orphan_df,
    [
        "relationship",
        "_relationship"
    ]
)

orphan_issue_column = find_column_case_insensitive(
    orphan_df,
    [
        "issue_type",
        "_issue_type",
        "_quarantine_type"
    ]
)

print("=" * 65)
print("ORPHAN RECORD ANALYSIS")
print("=" * 65)

print("Total orphan records:", orphan_count)

if orphan_relationship_column:

    print(
        "Relationship column:",
        orphan_relationship_column
    )

    orphan_relationship_summary_df = (
        orphan_df
        .groupBy(orphan_relationship_column)
        .count()
        .withColumnRenamed(
            orphan_relationship_column,
            "relationship"
        )
        .withColumnRenamed(
            "count",
            "orphan_record_count"
        )
    )

    display(
        orphan_relationship_summary_df.orderBy(
            "relationship"
        )
    )

else:

    print(
        "No relationship column found in orphan table."
    )

if orphan_issue_column:

    print("Issue type column:", orphan_issue_column)

    orphan_issue_summary_df = (
        orphan_df
        .groupBy(orphan_issue_column)
        .count()
        .orderBy(F.desc("count"))
    )

    display(orphan_issue_summary_df)

else:

    print("No issue type column found in orphan table.")

print("Orphan analysis completed.")


## 9. Analyze quarantine records

quarantine_relationship_column = find_column_case_insensitive(
    quarantine_df,
    [
        "_relationship",
        "relationship"
    ]
)

quarantine_issue_column = find_column_case_insensitive(
    quarantine_df,
    [
        "_quarantine_type",
        "issue_type",
        "_issue_type"
    ]
)

print("=" * 65)
print("QUARANTINE ANALYSIS")
print("=" * 65)

print("Total quarantine records:", quarantine_count)

if quarantine_relationship_column:

    print(
        "Relationship column:",
        quarantine_relationship_column
    )

    quarantine_relationship_summary_df = (
        quarantine_df
        .groupBy(quarantine_relationship_column)
        .count()
        .withColumnRenamed(
            quarantine_relationship_column,
            "relationship"
        )
        .withColumnRenamed(
            "count",
            "quarantine_record_count"
        )
    )

    display(
        quarantine_relationship_summary_df.orderBy(
            "relationship"
        )
    )

else:

    print(
        "No relationship column found in quarantine table."
    )

if quarantine_issue_column:

    print(
        "Quarantine issue column:",
        quarantine_issue_column
    )

    quarantine_issue_summary_df = (
        quarantine_df
        .groupBy(quarantine_issue_column)
        .count()
        .orderBy(F.desc("count"))
    )

    display(quarantine_issue_summary_df)

else:

    print(
        "No quarantine issue type column found."
    )

print("Quarantine analysis completed.")


## 10. Calculate relationship KPIs

relationship_kpi_df = (
    relationship_detail_df
    .agg(

        F.count("*").alias(
            "total_relationships"
        ),

        F.sum(
            "parent_records"
        ).alias(
            "total_parent_records"
        ),

        F.sum(
            "child_records"
        ).alias(
            "total_child_records"
        ),

        F.sum(
            "unique_parent_keys"
        ).alias(
            "total_unique_parent_keys"
        ),

        F.sum(
            "null_child_foreign_keys"
        ).alias(
            "total_null_foreign_keys"
        ),

        F.sum(
            "orphan_child_records"
        ).alias(
            "total_orphan_child_records"
        ),

        F.sum(
            "referenced_parent_records"
        ).alias(
            "total_referenced_parent_records"
        ),

        F.sum(
            "parents_without_children"
        ).alias(
            "total_parents_without_children"
        ),

        F.round(
            F.avg("reference_match_rate"),
            2
        ).alias(
            "average_reference_match_rate"
        ),

        F.round(
            F.avg("parent_usage_rate"),
            2
        ).alias(
            "average_parent_usage_rate"
        ),

        F.sum(
            "cardinality_key_count"
        ).alias(
            "total_cardinality_keys"
        )
    )
)

print("Relationship KPIs generated.")

display(relationship_kpi_df)


## 11. Calculate overall integrity metrics

overall_integrity_df = (
    relationship_detail_df
    .agg(

        F.sum(
            "child_records"
        ).alias(
            "total_child_records"
        ),

        F.sum(
            "orphan_child_records"
        ).alias(
            "total_orphan_records"
        ),

        F.sum(
            "null_child_foreign_keys"
        ).alias(
            "total_null_foreign_keys"
        )
    )

    .withColumn(
        "overall_referential_integrity_percentage",
        F.when(
            F.col("total_child_records") > 0,
            F.round(
                (
                    F.col("total_child_records")
                    - F.col("total_orphan_records")
                )
                / F.col("total_child_records") * 100,
                2
            )
        ).otherwise(100.0)
    )

    .withColumn(
        "overall_null_foreign_key_percentage",
        F.when(
            F.col("total_child_records") > 0,
            F.round(
                F.col("total_null_foreign_keys")
                / F.col("total_child_records") * 100,
                2
            )
        ).otherwise(0.0)
    )
)

print("Overall integrity metrics generated.")

display(overall_integrity_df)


## 12. Build combined dashboard detail

DASHBOARD_DETAIL_COLUMNS = [
    "relationship",
    "parent_dataset",
    "parent_key",
    "child_dataset",
    "child_key",
    "parent_records",
    "unique_parent_keys",
    "child_records",
    "null_child_foreign_keys",
    "orphan_child_records",
    "referenced_parent_records",
    "parents_without_children",
    "reference_match_rate",
    "parent_usage_rate",
    "orphan_percentage",
    "null_fk_percentage",
    "dashboard_status",

    "cardinality_key_count",
    "cardinality_distinct_child_key_count",
    "cardinality_child_record_count",
    "cardinality_maximum_children_per_key",
    "cardinality_average_children_per_key",
    "cardinality_minimum_children_per_key",

    "dashboard_run_id",
    "dashboard_generated_at"
]

dashboard_detail_df = (
    relationship_detail_df
    .select(*DASHBOARD_DETAIL_COLUMNS)
)

print("Combined dashboard detail generated.")

print("Dashboard detail schema:")

dashboard_detail_df.printSchema()

display(
    dashboard_detail_df.orderBy("relationship")
)


## 13. Build dashboard summary

dashboard_summary_df = (
    relationship_detail_df
    .agg(

        F.count("*").alias(
            "total_relationships"
        ),

        F.sum(
            F.when(
                F.col("dashboard_status") == "HEALTHY",
                1
            ).otherwise(0)
        ).alias(
            "healthy_relationships"
        ),

        F.sum(
            F.when(
                F.col("dashboard_status") == "ORPHANS_FOUND",
                1
            ).otherwise(0)
        ).alias(
            "relationships_with_orphans"
        ),

        F.sum(
            F.when(
                F.col("dashboard_status") == "NULL_FOREIGN_KEYS_FOUND",
                1
            ).otherwise(0)
        ).alias(
            "relationships_with_null_foreign_keys"
        ),

        F.sum(
            "parent_records"
        ).alias(
            "total_parent_records"
        ),

        F.sum(
            "child_records"
        ).alias(
            "total_child_records"
        ),

        F.sum(
            "unique_parent_keys"
        ).alias(
            "total_unique_parent_keys"
        ),

        F.sum(
            "null_child_foreign_keys"
        ).alias(
            "total_null_foreign_keys"
        ),

        F.sum(
            "orphan_child_records"
        ).alias(
            "total_orphan_records"
        ),

        F.sum(
            "referenced_parent_records"
        ).alias(
            "total_referenced_parent_records"
        ),

        F.sum(
            "parents_without_children"
        ).alias(
            "total_parents_without_children"
        ),

        F.round(
            F.avg("reference_match_rate"),
            2
        ).alias(
            "average_reference_match_rate"
        ),

        F.round(
            F.avg("parent_usage_rate"),
            2
        ).alias(
            "average_parent_usage_rate"
        ),

        F.sum(
            "cardinality_key_count"
        ).alias(
            "total_cardinality_keys"
        ),

        F.sum(
            "cardinality_child_record_count"
        ).alias(
            "total_cardinality_child_records"
        )
    )

    .withColumn(
        "overall_referential_integrity_percentage",
        F.when(
            F.col("total_child_records") > 0,
            F.round(
                (
                    F.col("total_child_records")
                    - F.col("total_orphan_records")
                )
                / F.col("total_child_records") * 100,
                2
            )
        ).otherwise(100.0)
    )

    .withColumn(
        "overall_null_foreign_key_percentage",
        F.when(
            F.col("total_child_records") > 0,
            F.round(
                F.col("total_null_foreign_keys")
                / F.col("total_child_records") * 100,
                2
            )
        ).otherwise(0.0)
    )

    .withColumn(
        "total_orphan_table_records",
        F.lit(orphan_count)
    )

    .withColumn(
        "total_quarantine_records",
        F.lit(quarantine_count)
    )

    .withColumn(
        "dashboard_run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "dashboard_generated_at",
        F.lit(DASHBOARD_TIMESTAMP)
    )
)

print("Dashboard summary generated.")

display(dashboard_summary_df)


## 14. Save dashboard detail table

(
    dashboard_detail_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_DETAIL_TABLE)
)

print("Dashboard detail table saved:")
print(DASHBOARD_DETAIL_TABLE)


## 15. Save dashboard summary table

(
    dashboard_summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_SUMMARY_TABLE)
)

print("Dashboard summary table saved:")
print(DASHBOARD_SUMMARY_TABLE)


## 16. Validate dashboard output tables

saved_detail_df = spark.table(
    DASHBOARD_DETAIL_TABLE
)

saved_summary_df = spark.table(
    DASHBOARD_SUMMARY_TABLE
)

saved_detail_count = saved_detail_df.count()
saved_summary_count = saved_summary_df.count()

print("=" * 70)
print("DASHBOARD OUTPUT VALIDATION")
print("=" * 70)

print(
    "Relationship report rows:",
    relationship_count
)

print(
    "Dashboard detail rows:",
    saved_detail_count
)

print(
    "Dashboard summary rows:",
    saved_summary_count
)

print(
    "Orphan source rows:",
    orphan_count
)

print(
    "Quarantine source rows:",
    quarantine_count
)

if saved_detail_count != relationship_count:
    raise Exception(
        "Dashboard detail row count does not match relationship report."
    )

if saved_summary_count != 1:
    raise Exception(
        "Dashboard summary must contain exactly one row."
    )

print("\nOutput validation passed.")


## 17. Display final dashboard results

print("=" * 75)
print("REFERENTIAL INTEGRITY DASHBOARD - FINAL RESULTS")
print("=" * 75)

print("\n1. Relationship dashboard detail")

display(
    spark.table(
        DASHBOARD_DETAIL_TABLE
    ).orderBy("relationship")
)

print("\n2. Overall dashboard summary")

display(
    spark.table(
        DASHBOARD_SUMMARY_TABLE
    )
)

print("\n3. Relationship status distribution")

status_summary_df = (
    spark.table(DASHBOARD_DETAIL_TABLE)
    .groupBy("dashboard_status")
    .count()
    .orderBy("dashboard_status")
)

display(status_summary_df)

print("\n4. Cardinality statistics")

display(
    spark.table(DASHBOARD_DETAIL_TABLE)
    .select(
        "relationship",
        "cardinality_key_count",
        "cardinality_distinct_child_key_count",
        "cardinality_child_record_count",
        "cardinality_maximum_children_per_key",
        "cardinality_average_children_per_key",
        "cardinality_minimum_children_per_key"
    )
    .orderBy("relationship")
)

print("=" * 75)
print("NOTEBOOK 06.4 COMPLETED SUCCESSFULLY")
print("=" * 75)

print("Dashboard detail:", DASHBOARD_DETAIL_TABLE)
print("Dashboard summary:", DASHBOARD_SUMMARY_TABLE)
print("Run ID:", RUN_ID)


### Query 1 — Overall integrity KPIs

In [0]:
%sql

SELECT
    total_relationships,
    healthy_relationships,
    relationships_with_orphans,
    relationships_with_null_foreign_keys,
    total_orphan_records,
    total_null_foreign_keys,
    overall_referential_integrity_percentage,
    overall_null_foreign_key_percentage,
    total_orphan_table_records,
    total_quarantine_records
FROM workspace.ecommerce_silver.referential_integrity_dashboard_summary;


### Check the actual table schema

In [0]:
%sql

DESCRIBE TABLE
workspace.ecommerce_silver.referential_integrity_dashboard_summary;


### Query 2 — Relationship health

In [0]:
%sql

SELECT
    relationship,
    parent_dataset,
    child_dataset,
    parent_records,
    child_records,
    orphan_child_records,
    null_child_foreign_keys,
    reference_match_rate,
    dashboard_status
FROM workspace.ecommerce_silver.referential_integrity_dashboard_detail
ORDER BY orphan_child_records DESC;


### Query 3 — Relationships with issues

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_silver.referential_integrity_dashboard_detail
WHERE dashboard_status != 'HEALTHY';


### Query 4 — Parent records without children

In [0]:
%sql

SELECT
    relationship,
    parent_dataset,
    parents_without_children,
    parent_usage_rate
FROM workspace.ecommerce_silver.referential_integrity_dashboard_detail
ORDER BY parent_usage_rate ASC;